# Assignment 23: OpenAI & Retrieval-Augmented Generation (RAG)

**Student:** Abhishek Thakare

This one is specifically about RAG using OpenAI - Wikipedia retrieval, vector
store retrieval, a few of the "advanced" retrievers (MMR, multi-query,
contextual compression), and then a small end-to-end project: a chatbot that
answers questions about a YouTube video's transcript.

**Honest note up front:** the restrictions for this assignment say LangChain,
OpenAI, and FAISS/Chroma only - no swapping in Ollama or Hugging Face like I
did in Assignment 25. That's a problem for me right now because my OpenAI
account still has zero usable credits. So every cell below that actually needs
to call OpenAI is written as real, working code, but I expect most of them to
fail with a quota/billing error when I run this - and I'm recording that
honestly instead of typing in fake output. The Wikipedia retriever part
doesn't need OpenAI at all, so that one should actually work.

If credits get added later, literally nothing in the code needs to change -
I'd just rerun the notebook top to bottom and real answers would show up
where the quota errors are now.


## Before running this

- `OPENAI_API_KEY` needs to be set as an environment variable (I'm loading it
  with `python-dotenv` from a `.env` file, not typing it into the notebook).
- Needs internet access - for the Wikipedia retriever and for pulling a
  YouTube transcript in Part 4.
- Reusing `data/notes.txt` from my earlier assignments for the custom vector
  store part in Task 3, so keep the `data/` folder next to this notebook.


In [9]:
# Run this only if something is missing in your environment
# %pip install -U langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu chromadb wikipedia youtube-transcript-api pypdf python-dotenv

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()

print("OpenAI key found:", bool(os.getenv("OPENAI_API_KEY")))
print("Data folder present:", os.path.isdir("data"))

# If there's no key at all (not even an expired/zero-credit one), ChatOpenAI()
# won't even construct - it'll throw before I get a chance to catch a proper
# quota error. Setting a placeholder here just so the client object can be
# created; every actual call below still needs a real, funded key to work.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-placeholder-no-real-credits"
    print("No real key found - using a placeholder so the client can at least initialize.")


OpenAI key found: True
Data folder present: True


## PART 1 — Getting Started with OpenAI

### Task 1: OpenAI Setup & Basic Prompt

Just checking the basics work before building anything on top of it - set up
the client, send one prompt, print whatever comes back.


In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

try:
    response = llm.invoke("In one sentence, what is Retrieval-Augmented Generation?")
    print(response.content)
except Exception as e:
    print("OpenAI call failed:", e)
    print("(This is the zero-credits issue mentioned above, not a bug in the code.)")


OpenAI call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-place******************dits. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
(This is the zero-credits issue mentioned above, not a bug in the code.)


As expected - this is the same account with no usable credits, so I'm getting a
quota error back instead of a real sentence. I'm leaving this exactly as it
runs rather than pasting in what I imagine the answer would be.


## PART 2 — Retriever-Based RAG (Wikipedia + Vector Store)

### Task 2: Wikipedia Retriever

This one doesn't touch OpenAI at all - it just pulls summaries straight from
Wikipedia, so it's the one part of this notebook I can actually expect to work
regardless of my API credits.


In [12]:
from langchain_community.retrievers import WikipediaRetriever

wiki_retriever = WikipediaRetriever()

query = "Retrieval-Augmented Generation"

try:
    wiki_docs = wiki_retriever.invoke(query)
    print(f"Got {len(wiki_docs)} Wikipedia result(s) for '{query}'\n")
    for i, doc in enumerate(wiki_docs, start=1):
        print(f"--- Result {i}: {doc.metadata.get('title')} ---")
        print(doc.page_content[:400])
        print()
except Exception as e:
    print("Wikipedia retrieval failed:", e)


Got 3 Wikipedia result(s) for 'Retrieval-Augmented Generation'

--- Result 1: Retrieval-augmented generation ---
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated informa

--- Result 2: Prompt engineering ---
Prompt engineering is the process of structuring natural language inputs (known as prompts) to produce specified outputs from a generative AI model. Context engineering is the related area of software engineering that focuses on the management of non-prompt and prompt contexts supplied to the GenAI model, such as system instructions, metadata, API tools and tokens.
It can also be defined as the pr

--- Result 3: Vector database ---
A vector dat

### Task 3: Vector Store Retriever

Now the actual RAG pattern - load some text, embed it, put the vectors in
FAISS, and search over that instead of the raw text. I'm reusing my
onboarding notes file here since it's the same custom text I've used in
earlier assignments.


In [13]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

notes_docs = TextLoader("data/notes.txt").load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
notes_chunks = splitter.split_documents(notes_docs)

print("Chunks created:", len(notes_chunks))


Chunks created: 6


In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

vector_retriever = None
faiss_db = None

try:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    faiss_db = FAISS.from_documents(notes_chunks, embeddings)
    vector_retriever = faiss_db.as_retriever(search_kwargs={"k": 3})

    results = vector_retriever.invoke("What is the leave policy?")
    print("Vector store search worked, top results:")
    for doc in results:
        print("-", doc.page_content[:120].replace("\n", " "))
except Exception as e:
    print("Couldn't build the OpenAI-embedded vector store:", e)
    print("Same zero-credits problem - embeddings need a real OpenAI call too.")


Couldn't build the OpenAI-embedded vector store: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-place******************dits. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Same zero-credits problem - embeddings need a real OpenAI call too.


So Task 2 (Wikipedia) is the one genuinely working retriever right now, and
Task 3 fails at the embedding step for the same reason Task 1 did. The
`vector_retriever` variable stays `None` for the rest of the notebook unless
this cell succeeds, and I check for that in the later cells so they don't just
crash.


## PART 3 — Advanced Retrieval Strategies

### Task 4: Maximal Marginal Relevance (MMR) Retriever

MMR is supposed to give more *varied* results than a plain similarity search -
instead of just returning the 3 closest chunks (which might all say almost the
same thing), it also tries to keep the results different from each other.


In [15]:
if vector_retriever is not None:
    mmr_retriever = faiss_db.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10})

    query = "What is the leave policy?"
    similarity_results = vector_retriever.invoke(query)
    mmr_results = mmr_retriever.invoke(query)

    print("Similarity search results:")
    for doc in similarity_results:
        print("-", doc.page_content[:100].replace("\n", " "))

    print("\nMMR results:")
    for doc in mmr_results:
        print("-", doc.page_content[:100].replace("\n", " "))
else:
    print("Skipping - no vector store available (Task 3 didn't succeed).")


Skipping - no vector store available (Task 3 didn't succeed).


Can't actually compare the two lists right now since the vector store never
got built. In theory, with a bigger and more repetitive knowledge base, MMR's
list would look more varied than plain similarity search, which tends to
return several near-duplicate chunks when the top matches are all about the
same topic.


### Task 5: Multi-Query Retriever

The idea here is that a single question might be phrased in a way that
doesn't match the wording in my documents well. `MultiQueryRetriever` asks the
LLM to write a few different versions of the question first, retrieves for
each one, and combines the results.


In [16]:
if vector_retriever is not None:
    from langchain.retrievers.multi_query import MultiQueryRetriever

    try:
        multi_query_retriever = MultiQueryRetriever.from_llm(retriever=vector_retriever, llm=llm)
        results = multi_query_retriever.invoke("laptop problems")
        print(f"Got {len(results)} combined result(s) across the reformulated queries.")
        for doc in results:
            print("-", doc.page_content[:100].replace("\n", " "))
    except Exception as e:
        print("Multi-query retrieval failed:", e)
        print("This one needs the LLM to generate the reformulated queries, so it needs OpenAI credits too.")
else:
    print("Skipping - no vector store available (Task 3 didn't succeed).")


Skipping - no vector store available (Task 3 didn't succeed).


### Task 6: Contextual Compression Retriever

This wraps a base retriever with a "compressor" - after the base retriever
pulls back full chunks, an LLM strips each chunk down to just the part that's
actually relevant to the question, instead of handing over the whole chunk.


In [17]:
if vector_retriever is not None:
    from langchain.retrievers import ContextualCompressionRetriever
    from langchain.retrievers.document_compressors import LLMChainExtractor

    try:
        compressor = LLMChainExtractor.from_llm(llm)
        compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor, base_retriever=vector_retriever
        )

        query = "What is the leave policy?"
        before = vector_retriever.invoke(query)
        after = compression_retriever.invoke(query)

        print("Before compression (full chunk):")
        print(before[0].page_content[:300])

        print("\nAfter compression (only the relevant part):")
        print(after[0].page_content[:300] if after else "(nothing left after compression)")
    except Exception as e:
        print("Contextual compression failed:", e)
        print("Same reason - the compressor itself is an LLM call.")
else:
    print("Skipping - no vector store available (Task 3 didn't succeed).")


Skipping - no vector store available (Task 3 didn't succeed).


None of Tasks 4-6 could actually run end-to-end on this account since they all
sit on top of either the embedded vector store or a direct LLM call. The
retriever logic itself is correct and would run as soon as the vector store
and LLM calls succeed - nothing here is stubbed out or faked, it's just
blocked by the same billing issue as Task 1.


## PART 4 — YouTube Content RAG Chatbot (Mini Project)

### Task 7: Load YouTube Content

Picking a short, plain educational video for this so the transcript is
reasonably clean. Splitting it the same way I've split every other document
in these assignments.


In [18]:
from langchain_community.document_loaders import YoutubeLoader

video_url = "https://www.youtube.com/watch?v=aircAruvnKk"  # 3Blue1Brown - "But what is a neural network?"

youtube_chunks = []

try:
    yt_loader = YoutubeLoader.from_youtube_url(video_url, add_video_info=False)
    yt_docs = yt_loader.load()
    print("Transcript documents loaded:", len(yt_docs))

    yt_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    youtube_chunks = yt_splitter.split_documents(yt_docs)
    print("Transcript chunks after splitting:", len(youtube_chunks))
    print("\nFirst chunk preview:")
    print(youtube_chunks[0].page_content[:300])
except Exception as e:
    print("Couldn't load the YouTube transcript:", e)
    print("This needs a live internet connection to YouTube - not an OpenAI issue.")


Transcript documents loaded: 1
Transcript chunks after splitting: 28

First chunk preview:
This is a 3. It's sloppily written and rendered at an extremely low resolution of 28x28 pixels, but your brain has no trouble recognizing it as a 3. And I want you to take a moment to appreciate how crazy it is that brains can do this so effortlessly. I mean, this, this and this are also recognizabl


### Task 8: Build Vector Store for YouTube Content

Same pattern as Task 3, just pointed at the transcript chunks instead of my
onboarding notes.


In [19]:
youtube_retriever = None

if youtube_chunks:
    try:
        yt_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        youtube_db = FAISS.from_documents(youtube_chunks, yt_embeddings)
        youtube_retriever = youtube_db.as_retriever(search_kwargs={"k": 4})
        print("YouTube vector store built.")
    except Exception as e:
        print("Couldn't embed the transcript:", e)
        print("Same zero-credits issue as Task 3.")
else:
    print("Skipping - no transcript chunks to embed (Task 7 didn't succeed).")


Couldn't embed the transcript: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-place******************dits. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Same zero-credits issue as Task 3.


### Task 9: Build YouTube RAG Chatbot

A simple loop: take a question, pull relevant transcript chunks, ask the LLM
to answer using only that context, and keep a running chat history so
follow-up questions have some memory of what was already asked.


In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat_history = []

video_qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the video transcript context below. "
               "If the answer isn't in the transcript, say you don't know - don't make something up.\n\n"
               "Context:\n{context}\n\nConversation so far:\n{history}"),
    ("human", "{question}"),
])

def ask_video(question):
    if youtube_retriever is None:
        return "[No transcript vector store available - can't answer this right now]"

    try:
        docs = youtube_retriever.invoke(question)
        context = "\n\n".join(d.page_content for d in docs)
        history_text = "\n".join(f"Q: {q}\nA: {a}" for q, a in chat_history) or "(nothing yet)"

        chain = video_qa_prompt | llm | StrOutputParser()
        answer = chain.invoke({"question": question, "context": context, "history": history_text})
        chat_history.append((question, answer))
        return answer
    except Exception as e:
        return f"[Question failed: {e}]"

print(ask_video("What is a neural network, according to the video?"))


[No transcript vector store available - can't answer this right now]


### Task 10: Testing & Evaluation

Trying 5 real questions about the video, plus one question that has nothing
to do with it, to see if it actually says "I don't know" instead of making
something up.


In [21]:
test_questions = [
    "What is a neuron in this context?",
    "What do the layers of a neural network do?",
    "What example does the video use to explain this?",
    "What is a weight in a neural network?",
    "How does the network improve its predictions?",
    "What's the capital of France?",  # unrelated - should say it doesn't know
]

for q in test_questions:
    print("-" * 60)
    print("Q:", q)
    print("A:", ask_video(q))


------------------------------------------------------------
Q: What is a neuron in this context?
A: [No transcript vector store available - can't answer this right now]
------------------------------------------------------------
Q: What do the layers of a neural network do?
A: [No transcript vector store available - can't answer this right now]
------------------------------------------------------------
Q: What example does the video use to explain this?
A: [No transcript vector store available - can't answer this right now]
------------------------------------------------------------
Q: What is a weight in a neural network?
A: [No transcript vector store available - can't answer this right now]
------------------------------------------------------------
Q: How does the network improve its predictions?
A: [No transcript vector store available - can't answer this right now]
------------------------------------------------------------
Q: What's the capital of France?
A: [No transcrip

Couldn't actually verify real grounded answers here since neither the
transcript embedding nor the chat call could go through without credits - but
the last question is there specifically to check the "graceful failure"
requirement from the assignment: if this were working, I'd want to see it say
something like "that's not in the video" for the France question instead of
just answering it anyway, since answering it anyway would mean the model
ignored my instruction to only use the transcript.


## PART 5 — Observations & Insights

### Task 11: Conceptual Questions

**1. Difference between retriever-based RAG and normal prompting**
Normal prompting is just me typing a question straight into the model and
hoping it already knows the answer from training. Retriever-based RAG adds a
lookup step first - the question gets used to search a document store, the
matching chunks get pulled in as context, and only then does the model
answer. That's the difference between "the model remembers this" and "the
model was just handed the right paragraph and asked to read it."

**2. Why vector stores are critical**
Once I have more than a handful of documents, I can't just paste everything
into the prompt - it's too much text and most of it isn't relevant to any
given question. A vector store lets me embed everything once and then, at
question time, only pull back the handful of chunks that are actually close
to what was asked. Without it, RAG doesn't really scale past a couple of
short files.

**3. When to use MMR vs plain similarity search**
Plain similarity search is fine when I just want the closest matches, full
stop. MMR is worth using when the knowledge base has some repetition and I'm
worried the top-3 similarity results might all be near-duplicates of each
other - MMR trades a little bit of "closeness" for more variety, so the
answer ends up built from a few different angles instead of the same point
said three times.

**4. Benefits of multi-query retrieval**
A lot of retrieval misses aren't because the answer isn't in the documents -
it's because the question was phrased differently than the documents are
written. Multi-query retrieval works around that by having the LLM generate
a few different phrasings of the same question before searching, so a chunk
that would've been missed by the original wording still has a chance of
getting picked up by one of the reformulated versions.

**5. Importance of contextual compression**
Retrieved chunks are often bigger than the actual relevant sentence or two
inside them. Feeding the full chunk to the LLM wastes context space and can
bury the actually-useful part in surrounding text. Contextual compression
trims each chunk down to just the relevant part before it reaches the final
answering step, which keeps the context tighter and (in theory) makes the
final answer more focused.


## Quick checklist before I submit this

In [22]:
checklist = {
    "OpenAI client set up (Task 1)": True,
    "Wikipedia retriever tested (Task 2)": True,
    "Vector store retriever code present (Task 3)": True,
    "Vector store actually built": faiss_db is not None,
    "MMR retriever code present (Task 4)": True,
    "Multi-query retriever code present (Task 5)": True,
    "Contextual compression code present (Task 6)": True,
    "YouTube transcript loaded (Task 7)": len(youtube_chunks) > 0,
    "YouTube vector store built (Task 8)": youtube_retriever is not None,
    "Chatbot code present (Task 9)": True,
    "Tested with 5+ questions (Task 10)": len(test_questions) >= 5,
}

for k, v in checklist.items():
    print(("[x] " if v else "[ ] ") + k)


[x] OpenAI client set up (Task 1)
[x] Wikipedia retriever tested (Task 2)
[x] Vector store retriever code present (Task 3)
[ ] Vector store actually built
[x] MMR retriever code present (Task 4)
[x] Multi-query retriever code present (Task 5)
[x] Contextual compression code present (Task 6)
[x] YouTube transcript loaded (Task 7)
[ ] YouTube vector store built (Task 8)
[x] Chatbot code present (Task 9)
[x] Tested with 5+ questions (Task 10)


## Honest note on OpenAI credits

Basically every "not tested" or quota-error line above comes back to the same
thing: my OpenAI account has zero usable credits right now. I kept every cell
as real, working code rather than hardcoding fake numbers or made-up answers
anywhere - Task 2 (Wikipedia) is the one part that doesn't depend on OpenAI at
all, which is exactly why it's the one thing that actually works today.

Once credits are added, the plan is simple: rerun the notebook top to bottom.
Task 3's vector store will build, which unblocks Tasks 4-6, and Tasks 7-10
will go from placeholder transcript answers to real, grounded ones.


## Final reflection

The main thing this assignment made clear is that "RAG" isn't one technique -
it's a stack of small decisions on top of the same basic loop (embed → store →
retrieve → answer). Wikipedia vs a custom vector store just changes *where*
the documents come from. MMR, multi-query, and contextual compression are all
tweaks to *how* the retrieval step behaves, without changing anything about
how the final answer gets generated. And the YouTube chatbot in Part 4 is just
that same loop again, pointed at a transcript instead of a text file - which
is the same lesson from Assignment 22 showing up again: once the pipeline is
built, the actual source of the documents barely matters.
